# Communication Primitives Deep Dive

## Overview

Understanding collective communication operations used in distributed training.

### Topics Covered
- AllReduce, AllGather, ReduceScatter
- Ring vs Tree algorithms
- Bandwidth analysis

## 1. Collective Operations

```
AllReduce: Reduce + Broadcast
GPU0: [1,2] ─┐         ┌─> [10,20]
GPU1: [3,4] ─┼─ SUM ───┼─> [10,20]
GPU2: [6,14]─┘         └─> [10,20]

AllGather: Gather to all
GPU0: [A] ─┐    ┌─> [A,B,C]
GPU1: [B] ─┼────┼─> [A,B,C]
GPU2: [C] ─┘    └─> [A,B,C]

ReduceScatter: Reduce + Scatter
GPU0: [1,2,3] ─┐    ┌─> [12] (sum of first elements)
GPU1: [4,5,6] ─┼────┼─> [15] (sum of second elements)
GPU2: [7,8,9] ─┘    └─> [18] (sum of third elements)
```

In [ ]:
def analyze_communication(data_size_gb, num_gpus, bandwidth_gbps=100):
    """Analyze communication time for different operations."""
    
    # Ring AllReduce: 2*(N-1)/N * data_size
    allreduce_volume = 2 * (num_gpus - 1) / num_gpus * data_size_gb
    allreduce_time = allreduce_volume / (bandwidth_gbps / 8) * 1000  # ms
    
    # AllGather: (N-1)/N * data_size
    allgather_volume = (num_gpus - 1) / num_gpus * data_size_gb
    allgather_time = allgather_volume / (bandwidth_gbps / 8) * 1000
    
    print(f"Data: {data_size_gb} GB, GPUs: {num_gpus}, BW: {bandwidth_gbps} Gbps")
    print(f"AllReduce: {allreduce_time:.1f} ms")
    print(f"AllGather: {allgather_time:.1f} ms")

analyze_communication(1.0, 8, 100)

## 2. Summary

| Operation | Volume | Use Case |
|-----------|--------|----------|
| AllReduce | 2(N-1)/N | DDP gradient sync |
| AllGather | (N-1)/N | FSDP param gather |
| ReduceScatter | (N-1)/N | FSDP gradient scatter |